<!-- SPDX-License-Identifier: CC-BY-4.0 -->
<!-- Copyright Contributors to the ODPi Egeria project 2024. -->

![Egeria Logo](https://raw.githubusercontent.com/odpi/egeria/main/assets/img/ODPi_Egeria_Logo_color.png)

### Egeria Workbook
-----

# Cataloguing and surveying PostgreSQL Servers

## Introduction

A [PostgreSQL server](https://www.postgresql.org/) is an open source technology that supports multiple relational databases.  The contents of these databases are organized into schemas, tables and columns giving a 4 level name space.  The default schema for a database is called **public** and there is also a special database called **pgcatalog** that provides a metadata catalog for the other databases managed by the servers.  PostgreSQL is a mature, fast and flexible database that is widely use in the industry.   Egeria itself uses a PostgreSQL database as an option for its metadata repository, and for the management of its observability data.

Egeria also has support for understanding and governing postgreSQL databases whatever their use.  This notebook takes you through these capabilities.

First lets initialize **pyegeria**.

In [5]:
# Initialize pyegeria

import os
view_server = os.environ.get("EGERIA_VIEW_SERVER","fs-view-server")
url = os.environ.get("EGERIA_VIEW_SERVER_URL","https://host.docker.internal:9443")
user_id = os.environ.get("EGERIA_USER", None)
user_pwd = os.environ.get("EGERIA_USER_PASSWORD")
egeria_width = 150

print("\n")
print("Accessing view server " + view_server + " on platform " + url + " for user " + user_id)
print("\n")

from pyegeria import load_mermaid, render_mermaid
load_mermaid()

from commands.ops.monitor_daemon_status import display_integration_daemon_status
from commands.ops.monitor_engine_activity import display_engine_activity_c
from commands.ops.monitor_engine_status import display_gov_eng_status

from datetime import datetime
import json
import time

from IPython.display import display, Markdown




Accessing view server qs-view-server on platform https://host.docker.internal:9443 for user erinoverview




In [6]:
from pyegeria import EgeriaTech

egeria_tech = EgeriaTech(view_server, url, user_id, user_pwd)
token = egeria_tech.create_egeria_bearer_token()


---

## Loading support for PostgreSQL Servers

The definition of the postgres connectors, templates and associated reference data are loaded via a [Content Pack](https://egeria-project.org/content-packs/) called `PostgresContentPack.omarchive`.  The content pack can be loaded multiple times without ill-effect so run the following command to make sure it is loaded.

---

In [11]:

elements = egeria_tech.find_elements_by_property_value(property_value="PostgreSQL", property_names=['displayName'], metadata_element_type_name="GovernanceActionProcess")
if type(elements) == str:
    print (elements)
else:
    for element in elements:
        if element:
            properties=element.get('properties')
            if properties:
                qualifiedName=properties.get('qualifiedName')
                description=properties.get('description')
                print('* ' + qualifiedName + ' - ' + description)



* PostgreSQLDatabase:DeleteAssetWithTemplateGovernanceActionProcess - Delete the asset for PostgreSQL Relational Database using the same template properties that were used to create it.  This will delete all of the metadata anchored to the asset and relationships to other entities such as the catalog target relationships.
* PostgreSQLDatabase:CreateAndSurveyGovernanceActionProcess - Create a PostgreSQL Relational Database, run a survey against it, and print out the resulting report.
* PostgreSQLServer::CreateAsCatalogTargetGovernanceActionProcess - Create a PostgreSQL Server and configure an integration connector to catalog its contents.
* PostgreSQLServer:DeleteAssetWithTemplateGovernanceActionProcess - Delete the asset for PostgreSQL Server using the same template properties that were used to create it.  This will delete all of the metadata anchored to the asset and relationships to other entities such as the catalog target relationships.
* PostgreSQLDatabaseSchema::CreateAsCatalogTa

----

## Survey a PostgreSQL Server

Egeria's PostgreSQL support includes the ability to survey the contents of a PostgreSQL Server to discover the databases that is manages.  This command creates a description of the PostgreSQL Server and runs a survey to understand its contents.  A summary of the survey results can be found in /distribution-hub/surveys.

---

In [6]:

createAndSurveyProcessName="PostgreSQLServer:CreateAndSurveyGovernanceActionProcess"

process_guid = egeria_tech.get_element_guid_by_unique_name(createAndSurveyProcessName)
print(process_guid)

process_graph = egeria_tech.get_governance_process_graph(process_guid, output_format="JSON")
render_mermaid(process_graph.get("governanceActionProcessMermaidGraph"))


In [3]:
from pyegeria import AutomatedCuration
automated_curation=AutomatedCuration(view_server, url, user_id, user_pwd)
token = automated_curation.create_egeria_bearer_token()

In [7]:
# 5442
requestParameters = {
    "serverName" : "LocalPostgreSQL1",
    "hostIdentifier" : "localhost",
    "portNumber" : "5442",
    "secretsStorePathName" : "secrets/integration.omsecrets",
    "secretsCollectionName" : "PostgreSQL Server Secret",
    "versionIdentifier" : "1.0",
    "description" : "PostgreSQL database in egeria-workspaces."
}

instance_guid=egeria_tech.initiate_gov_action_process(createAndSurveyProcessName, None, None, None, requestParameters, None, None)

'beca335a-8365-4507-9819-2a7dbcd554c1'

In [ ]:
instance_guid='766cc2cd-f606-4a70-a8ad-f99df65f0dbd'

process_graph = egeria_tech.get_governance_process_graph(instance_guid)

render_mermaid(process_graph.get("governanceActionProcessMermaidGraph"))

----

This next command catalogs the databases it finds on the PostgreSQL Server.

----

In [4]:

createAndCatalogServerName="PostgreSQLServer::CreateAsCatalogTargetGovernanceActionProcess"

process_guid = egeria_tech.get_element_guid_by_unique_name(createAndSurveyServerName)
print(process_guid)

process_graph = egeria_tech.get_governance_process_graph(process_guid, output_format="JSON")
render_mermaid(process_graph.get("governanceActionProcessMermaidGraph"))


AttributeError: 'AutomatedCuration' object has no attribute 'get_element_guid_by_unique_name'

In [9]:

requestParameters = {
    "serverName" : "LocalPostgreSQL1",
    "hostIdentifier" : "localhost",
    "portNumber" : "5442",
    "secretsStorePathName" : "secrets/integration.omsecrets",
    "secretsCollectionName" : "PostgreSQL Server Secret",
    "versionIdentifier" : "1.0",
    "description" : "PostgreSQL database in egeria-workspaces."
}

instance_guid=egeria_tech.initiate_gov_action_process(createAndCatalogServerName, None, None, None, requestParameters, None, None)



'8b8653f8-2091-4fce-9c4f-376e1892dc32'

In [ ]:

process_graph = egeria_tech.get_governance_process_graph(instance_guid)
render_mermaid(process_graph.get("governanceActionProcessMermaidGraph"))


In [ ]:
display_integration_daemon_status(['PostgreSQLServerCataloguer', 'JDBCDatabaseCataloguer'], 
                                  integ_server="Coco Pharmaceuticals.qs-integration-daemon", 
                                  paging=True, 
                                  width=150)